# 01 — Canny-guided detail Policy Preview

Inspect this candidate's routing, edge support, bounded detail and CPU cost on a fixed
sample of calibration originals. This is a read-only pixel preview, not a detector
benefit test or policy-selection procedure. No export, training or inference runs here.

**Inputs:** v4.yaml, shared split table, original calibration images.

**Outputs:** In-memory tables and figures only.

**Prerequisites:** Original BDD sources under the shared data_root. No upstream notebook runs.

**Outline**

- [1. Configuration](#section-1)
- [2. Policy on Calibration Samples](#section-2)
- [3. Policy Summary](#section-3)


<a id="section-1"></a>

## 1. Configuration

Change SAMPLES/SHOW freely; policy parameters live only in the version YAML.

In [1]:
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from vehicle_detection.common import project_root, read_image, require_paths
from vehicle_detection.experiment_data import load_experiment, shared_splits
from vehicle_detection.preprocess import apply_recipe, canny_edges

ROOT = project_root()
os.chdir(ROOT)  # Run All behaves the same from any kernel working directory.
EXPERIMENT = "v4"
require_paths(
    config=ROOT / f"notebooks/experiments/{EXPERIMENT}/{EXPERIMENT}.yaml",
    shared=ROOT / "configs/data.yaml",
)
config = load_experiment(ROOT, EXPERIMENT)

SAMPLES = 64
SHOW = 4
shared, rows = shared_splits(ROOT)
data_root = ROOT / shared["data_root"]
require_paths(data_root=data_root)
SAMPLES = 64
SHOW = 8

<a id="section-2"></a>

## 2. Policy on Calibration Samples

Features are measured on the original pixels; only the routed operator changes the image.

Canny runs on Gaussian-smoothed grayscale at original resolution, with L2 gradients. The edge map guides a soft local-detail mask; it is not the detector input. Density is measured before dilation. See README for the exact equations and limitations.

In [2]:
calibration = [r for r in rows if r["role"] == "calibration"]
if not calibration or SAMPLES < 1 or SHOW < 0:
    raise ValueError("Require calibration images, SAMPLES >= 1 and SHOW >= 0")
picked = np.random.default_rng(config["seed"]).choice(
    len(calibration), min(SAMPLES, len(calibration)), replace=False
)
records, pairs = [], []
for index in picked:
    row = calibration[index]
    image = read_image(data_root / row["source_path"])
    start = time.perf_counter()
    output, applied, scores = apply_recipe(image, config["variant"], config["recipes"])
    elapsed = (time.perf_counter() - start) * 1000
    delta = np.abs(output.astype(np.int16) - image.astype(np.int16))
    records.append(
        {
            "id": row["id"],
            "applied": "+".join(a["name"] for a in applied),
            "ms": elapsed,
            "changed_fraction": float(np.mean(np.any(delta != 0, axis=2))),
            "max_change": int(delta.max()),
            **scores,
        }
    )
    # Display the first fixed samples, including Skip; never pick by measured benefit.
    if len(pairs) < SHOW:
        operator = config["recipes"]["canny_detail"]
        edges = canny_edges(
            image, operator["low_threshold"], operator["high_threshold"], operator["blur_sigma"]
        )
        pairs.append((row["id"], image, edges, output, bool(applied)))
preview = pd.DataFrame(records)
display(
    preview.groupby("applied", dropna=False).agg(
        images=("id", "size"),
        mean_ms=("ms", "mean"),
        p95_ms=("ms", lambda s: s.quantile(0.95)),
        edge_density=("edge_density", "mean"),
        changed_fraction=("changed_fraction", "mean"),
        max_change=("max_change", "max"),
    )
)
for sample_id, before, edges, after, applied in pairs:
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), layout="constrained")
    axes[0].imshow(before[:, :, ::-1])
    axes[1].imshow(edges, cmap="gray", vmin=0, vmax=255)
    axes[2].imshow(after[:, :, ::-1])
    difference = np.max(np.abs(after.astype(np.int16) - before.astype(np.int16)), axis=2)
    axes[3].imshow(difference, cmap="magma", vmin=0, vmax=operator["max_offset"])
    labels = ("Original", "Canny support", "Apply" if applied else "Skip", "Absolute change")
    for ax, label in zip(axes, labels, strict=True):
        ax.set(title=f"{sample_id} · {label}", xticks=[], yticks=[])
    plt.show()

,images,mean_ms,p95_ms,edge_density,changed_fraction,max_change
applied,,,,,,
canny_detail,64,41.465875,46.11952,0.022165,0.072682,12


C:\Users\MSI\AppData\Local\Temp\ipykernel_524\253051842.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


<a id="section-3"></a>

## 3. Policy Summary

In [3]:
print("Variant:", config["variant"], "| Recipes:", config["recipes"])
print(f"Applied to {(preview.applied != '').mean():.1%} of {len(preview)} calibration samples")
print("Export happens only in 02_train_prepare; final-test pixels stay sealed.")
print("Exploratory candidate only: no AP or improvement has been measured.")

Variant: adaptive_canny_detail | Recipes: {'adaptive_canny_detail': {'brightness_above': 0.04, 'brightness_below': 0.9, 'noise_max': 0.02, 'edge_density_above': 0.001, 'edge_density_below': 0.2}, 'canny_detail': {'low_threshold': 50, 'high_threshold': 150, 'blur_sigma': 1.0, 'detail_sigma': 1.0, 'amount': 0.5, 'max_offset': 12, 'mask_radius': 1}}
Applied to 100.0% of 64 calibration samples
Export happens only in 02_train_prepare; final-test pixels stay sealed.
Exploratory candidate only: no AP or improvement has been measured.
